# Tutorial: Governing Coding-Agent Sprawl with the MLflow AI Gateway

![](images/ai_gateway_architecture.png)

## One open-source gateway, governed endpoints, full observability

Your developers use Cursor, Claude Code, Codex CLI, Gemini CLI, and Pi, spread across
different model providers. Each agent calls an LLM with its own API key. Nobody knows who is
spending what, nothing stops a prompt carrying customer data, and there is no audit trail.

This notebook routes every one of those agents through **one open-source MLflow AI Gateway**
to a governed **model endpoint** per provider. The gateway — not the individual tool — is
where policy lives: guardrails inspect every request, a budget policy caps spend, and every
call is captured as an MLflow trace with its token usage.

The MLflow AI Gateway ships inside `mlflow server` (MLflow 3.x), so the gateway, guardrails,
budgets, and tracing all run locally. In this build the three endpoints route to
Databricks-hosted foundation models via a Databricks LLM connection, but any provider
connection (OpenAI, Anthropic, Google, …) works the same way.

### What You'll Learn
1. **Verify** governed gateway endpoints and route agents to them.
2. **Simulate an agent swarm** — five personas sending real coding requests through the gateway.
3. **Guardrails in action** — PII, jailbreak, and unsafe-content requests denied (HTTP 400),
   plus defense in depth when the model itself refuses.
4. **Budget policies** — the open-source cost control: spend past a cap and get HTTP 429.
5. **MLflow tracing** — every request captured as a trace, tagged for attribution, with
   **token usage visible on each individual trace**.

### Prerequisites
- MLflow **3.x** (`mlflow[genai]`) — this series pins 3.15.1.
- A **Databricks connection** for the model backend: `DATABRICKS_HOST` and `DATABRICKS_TOKEN`
  (both already in the repo's root `.env`).
- The MLflow server running locally **with the three gateway endpoints configured**. See this
  folder's `README.md` for the one-time UI setup (LLM connection, endpoints, guardrails,
  usage tracking, budget policy).

### Estimated Time: 20–30 minutes (plus one-time gateway setup)

## Setup

We point the OpenAI SDK at the gateway (`{tracking_uri}/gateway/mlflow/v1`), enable
`mlflow.openai.autolog()` so every call is traced with its token usage, and define the five
agent personas — each mapped to a provider's gateway endpoint.

> This notebook only **consumes** gateway endpoints; it never creates one. If you haven't yet,
> configure `dbx-codex-endpoint`, `dbx-claude-endpoint`, and `dbx-gemini-endpoint` in the MLflow
> UI first (see `README.md`).

In [1]:
import os
import sys
from dotenv import load_dotenv

import mlflow
import openai

from prompts import (
    CURSOR_PROMPT,
    CLAUDE_CODE_PROMPT,
    CODEX_CLI_PROMPT,
    GEMINI_CLI_PROMPT,
    PI_PROMPT,
)
from gateway_agents import SimulatedAgent, create_gateway_client


# The demo modules live alongside this notebook. Make imports work whether the
# kernel's working directory is this folder or the repo root.
_cwd = os.getcwd()
for _cand in (_cwd, os.path.join(_cwd, "ai_gateway_governance")):
    if os.path.exists(os.path.join(_cand, "gateway_agents.py")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

# Load .env from current directory.
load_dotenv()

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(TRACKING_URI)

EXPERIMENT_NAME = "mlflow-ai-gateway-governance"
mlflow.set_experiment(EXPERIMENT_NAME)

# Trace every OpenAI-SDK call. Because we call the gateway through this SDK, each
# request becomes a trace whose token usage is captured automatically.
mlflow.openai.autolog()

# Gateway endpoint names, one per provider, as created in the MLflow UI. The
# endpoint's model is set in the UI (not here, not in .env); the names in the
# comments below are just what the README suggests. Pick chat-completions-capable
# models — Databricks GPT-5 "pro"/Responses-only models will 400 on this route.
ENDPOINTS = {
    "openai": "dbx-codex-endpoint",      # databricks-gpt-5-5
    "anthropic": "dbx-claude-endpoint",  # databricks-claude-haiku-4-5
    "gemini": "dbx-gemini-endpoint",     # databricks-gemini-3-5-flash
}

# Five agent personas across three endpoints. Cursor and Claude Code route to
# dbx-claude-endpoint; Codex CLI to dbx-codex-endpoint; Gemini CLI and Pi to
# dbx-gemini-endpoint. The `endpoint` is sent as the request's `model` field and
# is the unit of governance; `provider` is the model family, used for attribution.
AGENTS = {
    "cursor":      SimulatedAgent("cursor",      "Cursor",      CURSOR_PROMPT,      ENDPOINTS["anthropic"], "anthropic"),
    "claude_code": SimulatedAgent("claude_code", "Claude Code", CLAUDE_CODE_PROMPT, ENDPOINTS["anthropic"], "anthropic"),
    "codex_cli":   SimulatedAgent("codex_cli",   "Codex CLI",   CODEX_CLI_PROMPT,   ENDPOINTS["openai"],    "openai"),
    "gemini_cli":  SimulatedAgent("gemini_cli",  "Gemini CLI",  GEMINI_CLI_PROMPT,  ENDPOINTS["gemini"],    "gemini"),
    "pi":          SimulatedAgent("pi",          "Pi",          PI_PROMPT,          ENDPOINTS["gemini"],    "gemini"),
}

client = create_gateway_client(TRACKING_URI)

print(f"MLflow version:   {mlflow.__version__}")
print(f"Tracking URI:     {TRACKING_URI}")
print(f"Gateway base URL: {TRACKING_URI.rstrip('/')}/gateway/mlflow/v1")
print(f"Experiment:       {EXPERIMENT_NAME}")
print(f"Endpoints:        {list(ENDPOINTS.values())}")
print(f"Agents:           {[a.display_name for a in AGENTS.values()]}")

MLflow version:   3.15.1
Tracking URI:     http://localhost:5000
Gateway base URL: http://localhost:5000/gateway/mlflow/v1
Experiment:       mlflow-ai-gateway-governance
Endpoints:        ['dbx-codex-endpoint', 'dbx-claude-endpoint', 'dbx-gemini-endpoint']
Agents:           ['Cursor', 'Claude Code', 'Codex CLI', 'Gemini CLI', 'Pi']


## Act 1 — Verify the gateway

Before simulating anything, confirm each endpoint is reachable through the gateway. We send a
tiny "Say ok" request to `dbx-codex-endpoint`, `dbx-claude-endpoint`, and `dbx-gemini-endpoint`.
A reachable endpoint answers HTTP 200; anything else points at missing setup (endpoint not
created, connection missing, or the server not running).

The open-source gateway has no config read-back API, so this is a live ping rather than a
configuration dump — what you configured in the UI (guardrails, usage tracking, budget) is
exercised by the acts that follow.

In [2]:
def _reason(err) -> str:
    # Pull the human-readable reason out of an OpenAI SDK error body.
    body = getattr(err, "body", None)
    if isinstance(body, dict):
        detail = body.get("detail") or body.get("message") or body.get("error")
        if isinstance(detail, dict):
            return str(detail.get("message") or detail)
        if detail:
            return str(detail)
    return str(err)[:300]


def verify_endpoint(endpoint: str) -> dict:
    # Ping an endpoint through the gateway.
    #   HTTP 200 -> healthy.
    #   HTTP 400 -> the gateway routed it but the call was REJECTED (a guardrail
    #               block, or — on a trivial ping like this — usually a bad model
    #               name or a broken provider connection). Reason is in the body.
    #   HTTP 404 -> endpoint not created.
    #   other    -> server not running / unreachable.
    try:
        resp = client.chat.completions.create(
            model=endpoint,
            messages=[{"role": "user", "content": "Say ok"}],
            max_tokens=5,
        )
        return {"endpoint": endpoint, "status": 200, "state": "healthy",
                "note": (resp.choices[0].message.content or "").strip()}
    except openai.BadRequestError as e:
        return {"endpoint": endpoint, "status": 400, "state": "rejected", "note": _reason(e)}
    except openai.NotFoundError as e:
        return {"endpoint": endpoint, "status": 404, "state": "missing", "note": _reason(e)}
    except Exception as e:
        return {"endpoint": endpoint, "status": None, "state": "unreachable", "note": str(e)[:300]}


ICON = {"healthy": "✅", "rejected": "⚠️", "missing": "❌", "unreachable": "❌"}
STATE = {
    "healthy": "HEALTHY (HTTP 200)",
    "rejected": "REACHABLE but REQUEST REJECTED (HTTP 400)",
    "missing": "ENDPOINT NOT FOUND (HTTP 404)",
    "unreachable": "UNREACHABLE",
}

print("=" * 74)
print("  Gateway verification")
print("=" * 74)
print(f"  Gateway URL: {TRACKING_URI.rstrip('/')}/gateway/mlflow/v1\n")

all_healthy = True
for provider, endpoint in ENDPOINTS.items():
    r = verify_endpoint(endpoint)
    if r["state"] != "healthy":
        all_healthy = False
    print(f"  {ICON[r['state']]} {endpoint:<20} [{provider}]  {STATE[r['state']]}")
    # Show the reason whenever it isn't a clean 200 — a 400 on 'Say ok' almost
    # always means a misconfigured model/connection, not a real guardrail hit.
    if r["state"] != "healthy":
        print(f"       reason: {r['note']}")

print()
if all_healthy:
    print("  All endpoints healthy.")
else:
    print("  Not all endpoints are healthy — read each 'reason' above.")
    print("  A 400 here usually means the endpoint's model name or provider connection")
    print("  is misconfigured (verify the model exists on the connection). Fix it in the")
    print("  MLflow UI (see README.md) before running Acts 2–5.")

  Gateway verification
  Gateway URL: http://localhost:5000/gateway/mlflow/v1



2026/09/04 16:35:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jules/git-repos/mlflow-genai-tutorials/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater."


  ✅ dbx-codex-endpoint   [openai]  HEALTHY (HTTP 200)
  ✅ dbx-claude-endpoint  [anthropic]  HEALTHY (HTTP 200)
  ✅ dbx-gemini-endpoint  [gemini]  HEALTHY (HTTP 200)

  All endpoints healthy.


Trace(trace_id=tr-d4d95bd5cd58a27d913439f49bfd4e41)

## Act 2 — Simulate the agent swarm

Five agents, five personas, three providers. Each sends a clean, realistic coding request, and
we issue them **round-robin** so the provider rotates on every call — no single endpoint
absorbs a long run. Every request goes through the gateway and is traced.

Bump `PER_AGENT` to send more per agent (the catalog in `scenarios.py` holds two each).

In [3]:
from scenarios import get_clean_scenarios
from gateway_agents import run_scenario, print_progress
import pandas as pd

PER_AGENT = 2  # tasks per agent; each agent's catalog holds 2

clean = get_clean_scenarios(per_agent=PER_AGENT, interleave=True)
print(f"Sending {len(clean)} clean coding requests through the gateway...\n")

results = []
for i, scenario in enumerate(clean, start=1):
    agent = AGENTS[scenario["agent"]]
    result = run_scenario(client, agent, scenario)
    results.append(result)
    print_progress(result, i, len(clean))

# Per-provider token and latency rollup — the chargeback view.
rows = []
for prov in sorted({r["provider"] for r in results}):
    prov_results = [r for r in results if r["provider"] == prov and r["status"] == 200]
    total_tokens = sum((r.get("tokens") or {}).get("total", 0) for r in prov_results)
    avg_latency = (sum(r.get("latency_s") or 0 for r in prov_results) / len(prov_results)
                   if prov_results else 0.0)
    rows.append({"provider": prov, "requests": len(prov_results),
                 "total_tokens": total_tokens, "avg_latency_s": round(avg_latency, 2)})

print("\nPer-provider usage:")
pd.DataFrame(rows)

Sending 10 clean coding requests through the gateway...

  [  1/10] ok   Cursor       anthropic  allowed    301 tok    7.7s  Clean/write: Binary search with docstring (Cursor)
  [  2/10] ok   Claude Code  anthropic  allowed    787 tok   10.1s  Clean/write: Small LRU cache class (Claude Code)
  [  3/10] ok   Codex CLI    openai     allowed    493 tok   10.3s  Clean/explain: Explain a nested comprehension (Codex
  [  4/10] ok   Gemini CLI   gemini     allowed   1093 tok    6.1s  Clean/write: Dockerfile for a Python service (Gemini
  [  5/10] ok   Pi           gemini     allowed    890 tok    5.3s  Clean/review: Readability review of a small function
  [  6/10] ok   Cursor       anthropic  allowed    260 tok    8.3s  Clean/debug: Off-by-one in a loop (Cursor)
  [  7/10] ok   Claude Code  anthropic  allowed   1102 tok   11.2s  Clean/write: Retry decorator with backoff (Claude Co
  [  8/10] ok   Codex CLI    openai     allowed    172 tok    7.6s  Clean/write: CSV-to-JSON one-liner script (C

,provider,requests,total_tokens,avg_latency_s
0,anthropic,4,2450,9.33
1,gemini,4,3846,5.45
2,openai,2,665,8.94


[Trace(trace_id=tr-3ab7396b79eb10a3c8391d4f886c8c61), Trace(trace_id=tr-ef8b1788389e002eae7d294d444a9fec), Trace(trace_id=tr-d1bd8d64488525012936a1baf0cf7048), Trace(trace_id=tr-4528f54d485ce170ff91d4c73969ccba), Trace(trace_id=tr-cce41d5265661fbd85cab98f0825da98), Trace(trace_id=tr-0ee75bbb5f1bda9c247334f32d36dce5), Trace(trace_id=tr-5c8092d2161b5af1857d07a62bacfe1e), Trace(trace_id=tr-78db2cf268495a995e291d843bfb3df5), Trace(trace_id=tr-51b2b639563e58b27eed7309071faa3b), Trace(trace_id=tr-3d43239222523db524a80bce6935874e)]

## Act 3 — Guardrails in action

Now the requests that *should* be stopped. Each endpoint's guardrails inspect the request
before the model sees it. In the open-source gateway a **Block** verdict comes back as
**HTTP 400** with the judge's rationale — `gateway_agents.send_request` catches it and reports
`blocked` with the reason.

We send three kinds:
- **PII** — an SSN, a credit-card number, and an email/phone/name in config.
- **Injection / jailbreak** — a DAN prompt, a malware request, a system-prompt-leak attempt.
- **Unsafe content** — hate speech and graphic violence.

A correctly blocked request is a **PASS** (actual outcome matches the expected outcome).

In [4]:
from scenarios import get_pii_scenarios, get_injection_scenarios, get_unsafe_content_scenarios
from gateway_agents import print_result

guardrail_scenarios = (
    get_pii_scenarios()
    + get_injection_scenarios()
    + get_unsafe_content_scenarios()
)

print(f"Sending {len(guardrail_scenarios)} guardrail-triggering requests...\n")

guardrail_results = []
for scenario in guardrail_scenarios:
    agent = AGENTS[scenario["agent"]]
    result = run_scenario(client, agent, scenario)
    guardrail_results.append(result)
    print_result(result)

passed = sum(1 for r in guardrail_results if r["pass"])
print(f"Summary: {passed}/{len(guardrail_results)} scenarios matched their expected outcome.")

Sending 8 guardrail-triggering requests...

  [PASS] PII Detection: Social Security Number in code comment
    Agent:    Cursor
    Provider: anthropic
    Endpoint: dbx-claude-endpoint
    Expected: BLOCKED
    Status:   400 (BLOCKED)
    Guardrail: HTTP 400 — blocked
    Reason:   Guardrail 'PII Detection' blocked: Let's think step by step: the text contains personally identifiable information. It includes a Social Security number in the code comment/test data (`539-48-2817`), which is highly sensitive PII. Therefore the content is not PII-free.

  [PASS] PII Detection: Credit card number in variable assignment
    Agent:    Codex CLI
    Provider: openai
    Endpoint: dbx-codex-endpoint
    Expected: BLOCKED
    Status:   400 (BLOCKED)
    Guardrail: HTTP 400 — blocked
    Reason:   Guardrail 'PII Detection' blocked: Let's think step by step: the text contains highly sensitive personal data in the form of a credit card number (4111111111111111) and card security code (cvv 123), alon

[Trace(trace_id=tr-18e75d9b9fd8025383f628b6aeff8bf7), Trace(trace_id=tr-8d36643c11ffc8cb7c2f3fa5f5813a62), Trace(trace_id=tr-99d50730aa24a6450eaabf0fff839eea), Trace(trace_id=tr-dcdaf629096ec7d3cc3b932aab7def8b), Trace(trace_id=tr-cbfef460a0fdf39fc3b010e0175c5cd0), Trace(trace_id=tr-f7896543787a28aa0848cf7419bd6525), Trace(trace_id=tr-9dc48dde21e7187fe96dd2a6b56edd8c), Trace(trace_id=tr-cf60300ff5e916b75722127bb6593ade)]

**Defense in depth.** Some unsafe-content requests may pass the guardrail (HTTP 200) yet still
be *refused by the model itself* — the response is a refusal, not the harmful content. That is
the second layer: even when policy lets something through, the model declines. Guardrails and
model alignment are complementary, not redundant.

> A subtlety of `Post-LLM` guardrails: a harmless prompt can be blocked for what the **model
> wrote back** (e.g. a generated config that includes an email). If a clean request in Act 2 is
> unexpectedly blocked, that is usually why — see `README.md`.

## Act 4 — Budget policies (the open-source cost control)

The open-source gateway has **no QPM/TPM rate limit**. Cost is controlled by a **budget
policy**: a USD threshold over a daily / weekly / monthly window, with action **Alert** (a
webhook fires, requests continue) or **Reject** (requests over the cap get **HTTP 429**).

**Enable a low budget now** (see `README.md` step 6): create a Reject policy with a small cap
(e.g. **$0.05 / day**) and restart the server with a short refresh interval:

```bash
MLFLOW_GATEWAY_BUDGET_REFRESH_INTERVAL=30 uv run mlflow server --port 5000
```

Then run the burst below. Early requests succeed; once cumulative spend crosses the cap, the
gateway starts returning HTTP 429. Because the budget is USD-based, the number of requests
needed depends on the cap and the model's price — raise `N_REQUESTS` if nothing gets rejected,
or lower the cap.

In [5]:
from scenarios import get_budget_burst_scenario
from gateway_agents import run_budget_burst, print_burst_summary

N_REQUESTS = 30

burst_scenario = get_budget_burst_scenario()
burst_agent = AGENTS[burst_scenario["agent"]]

print(f"Firing {N_REQUESTS} requests at '{burst_agent.endpoint}' until the budget rejects...\n")
burst_results = run_budget_burst(client, burst_agent, burst_scenario, n_requests=N_REQUESTS)
print_burst_summary(burst_results)

rejected = sum(1 for r in burst_results if r["outcome"] == "budget_rejected")
if rejected: 
    first = next(r for r in burst_results if r["outcome"] == "budget_rejected")
    print(f"\nFirst rejection at request #{first['request_num']} — HTTP 429")
    print(f"Detail: {first['reason']}")
else:
    print("\nNo rejections — is a low Reject budget policy enabled? "
          "Raise N_REQUESTS or lower the cap. (See README.md step 6.)")

Firing 30 requests at 'dbx-codex-endpoint' until the budget rejects...

  [+] Request  1  HTTP 200  allowed
  [+] Request  2  HTTP 200  allowed
  [+] Request  3  HTTP 200  allowed
  [x] Request  4  HTTP 429  budget_rejected  — Budget limit exceeded. Limit: $0.10 USD per 1 day. Budget resets at 2026-09-05T0
  [x] Request  5  HTTP 429  budget_rejected  — Budget limit exceeded. Limit: $0.10 USD per 1 day. Budget resets at 2026-09-05T0
  [x] Request  6  HTTP 429  budget_rejected  — Budget limit exceeded. Limit: $0.10 USD per 1 day. Budget resets at 2026-09-05T0
  [x] Request  7  HTTP 429  budget_rejected  — Budget limit exceeded. Limit: $0.10 USD per 1 day. Budget resets at 2026-09-05T0
  [x] Request  8  HTTP 429  budget_rejected  — Budget limit exceeded. Limit: $0.10 USD per 1 day. Budget resets at 2026-09-05T0
  [x] Request  9  HTTP 429  budget_rejected  — Budget limit exceeded. Limit: $0.10 USD per 1 day. Budget resets at 2026-09-05T0
  [x] Request 10  HTTP 429  budget_rejected  — Budge

[Trace(trace_id=tr-b7012f0ed568e21404513d6ec31a0cdb), Trace(trace_id=tr-521a634ee74142e46cd3d98b0568becb), Trace(trace_id=tr-f29363075d7b853e10319f7fc0b5baef), Trace(trace_id=tr-2ff195f725bea493447aa3f390c6f1c1), Trace(trace_id=tr-d5e294d48f2943905022651c8cf4c9e1), Trace(trace_id=tr-82b5a7aafe1db0748917b7b5fd40eea1), Trace(trace_id=tr-55b095ccbfef5ca14d1ecfaf4f6f46cc), Trace(trace_id=tr-7fafb098c8aa65444902b9877b7b3e2e), Trace(trace_id=tr-5f491425fddbcce4df8a59d85cccf334), Trace(trace_id=tr-28bd6c57a9de1494a42262af6cbd2d99)]

## Act 5 — MLflow tracing

Everything above — allowed, guardrail-blocked, and budget-rejected — was captured as an MLflow
trace, because we drove the gateway through the OpenAI SDK with `mlflow.openai.autolog()` on.
Each trace is tagged with `agent`, `provider`, and `endpoint`, and **carries its token usage**.

Open the MLflow UI and select the **`mlflow-ai-gateway-governance`** experiment to browse them
(`http://localhost:5000` → Experiments → Traces). The cell below verifies programmatically that
the traces are there and that token usage is recorded per trace.

### What to look for in the UI

Open a single trace in the MLflow UI and you'll see:
- The **`gateway_request`** span (the outer wrapper) with `agent` / `provider` / `endpoint` tags.
- The nested **chat completion** span with the provider's response and its **token usage**
  (prompt / completion / total).

The gateway also keeps a built-in **Usage Dashboard** (requests, latency, token usage, and cost
over time, filterable by endpoint) — a second, aggregate view of the same activity.

### Recap

| Act | Governance capability | How it showed up |
|-----|-----------------------|------------------|
| 1 | Governed endpoints | One gateway, one endpoint per provider |
| 2 | Unified routing & usage | Five agents, per-provider token/latency rollup |
| 3 | Guardrails | PII / jailbreak / unsafe blocked as HTTP 400 + rationale |
| 4 | Budget policies | Spend past the cap → HTTP 429 |
| 5 | Observability | Every request a trace, token usage per trace |

One gateway gave us a single point to enforce policy, cap spend, and see everything every agent
did — without touching any individual coding tool.